# 10 · Changing data: INSERT, UPDATE, MERGE, SCD Type 2, incremental loads (Spark)

**DML** (Data Manipulation Language) changes rows: `INSERT`, `UPDATE`, `DELETE`, `MERGE`.
**DDL** (Data Definition Language) changes tables: `CREATE`, `ALTER`, `DROP`.

This notebook works on **copies** named `prac_*`, so the 7 practice tables never change.
Each question has a setup cell that rebuilds its copy: run it first, and again to start over.

Data engineering words used here:
- **upsert** = update the row if the key exists, insert it if not (`MERGE`)
- **SCD Type 2** (slowly changing dimension) = keep the **history**: close the old row, add a new current row
- **incremental load** = load only the new rows since last time (using a **watermark**, e.g. the max date already loaded)
- **idempotent** = running the job twice gives the same result as running it once

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/10_dml_merge_scd.ipynb`](../../spark/practice/10_dml_merge_scd.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. INSERT a row  ·  🟢 easy

Add customer **41**, `Test User`, city `Austin`, signup `2024-12-01` to `prac_customers`. Then show `num_customers` (the row count).

*Expected result: 1 row · columns: `num_customers`*

**How it works:** `INSERT INTO table (columns) VALUES (...)` adds one row; the SELECT COUNT checks it is there.

In [ ]:
new_row = spark.createDataFrame([(41, "Test User", "Austin", "2024-12-01")],
                                "customer_id int, name string, city string, signup_date string") \
               .withColumn("signup_date", F.to_date("signup_date"))
prac_customers = customers.unionByName(new_row)
res = prac_customers.agg(F.count("*").alias("num_customers"))
res.show()

+-------------+
|num_customers|
+-------------+
|           41|
+-------------+

**Why / traps:** Always list the columns in INSERT. If someone adds a column later, your insert still works.

DataFrames are **immutable** (they never change). You make a new DataFrame. To change stored data you write files again, or use a table format such as Delta Lake / Apache Iceberg that supports INSERT/UPDATE/MERGE.

## Q2. UPDATE with a condition  ·  🟢 easy

In `prac_customers`, set `city` to `Unknown` where it is NULL. Then show `customer_id`, `name`, `city` of the changed rows.

*Expected result: 1 row · columns: `customer_id`, `name`, `city`*

**How it works:** `UPDATE ... SET city = 'Unknown' WHERE city IS NULL` changes only the matching rows.

In [ ]:
prac_customers = customers.withColumn("city", F.coalesce("city", F.lit("Unknown")))
res = prac_customers.filter("city = 'Unknown'").select("customer_id", "name", "city")
res.show()

+-----------+-----------+-------+
|customer_id|       name|   city|
+-----------+-----------+-------+
|         13|Lucas Lopez|Unknown|
+-----------+-----------+-------+

**Why / traps:** Write the WHERE first and test it as a SELECT. An UPDATE without WHERE changes **every** row.

## Q3. DELETE with NOT EXISTS  ·  🟢 easy

Delete from `prac_customers` everyone who **never ordered**. Then show `num_customers`.

*Expected result: 1 row · columns: `num_customers`*

**How it works:** `DELETE ... WHERE NOT EXISTS (order for this customer)` removes customers who never ordered.

In [ ]:
prac_customers = customers.join(orders, "customer_id", "left_semi")
res = prac_customers.agg(F.count("*").alias("num_customers"))
res.show()

+-------------+
|num_customers|
+-------------+
|           32|
+-------------+

**Why / traps:** `TRUNCATE TABLE` removes **all** rows fast (no WHERE). In Oracle it cannot be rolled back.

## Q4. Upsert with MERGE  ·  🟡 medium

`prac_customer_updates` has 4 rows: customers 1 and 2 changed, 41 and 42 are new. MERGE them into `prac_customers`: update `name` and `city` when the id exists, insert when not. Then show customers 1, 2, 41, 42.

*Expected result: 4 rows · columns: `customer_id`, `name`, `city`, `signup_date`*

**How it works:** MERGE matches source to target on the key: `WHEN MATCHED` → UPDATE the row, `WHEN NOT MATCHED` → INSERT it. One statement does both.

In [ ]:
updates = spark.createDataFrame(
    [(1, "Aarav Nguyen", "Denver", "2023-01-26"), (2, "Priya Silva-Rao", "Seattle", "2023-09-08"),
     (41, "Nina Patel", "Boston", "2024-12-05"), (42, "Omar Haddad", "Austin", "2024-12-06")],
    "customer_id int, name string, city string, signup_date string"
).withColumn("signup_date", F.to_date("signup_date"))

# upsert = (old rows whose key is NOT in updates) + (all update rows)
prac_customers = customers.join(updates, "customer_id", "left_anti").unionByName(updates)
res = prac_customers.filter(F.col("customer_id").isin(1, 2, 41, 42)).orderBy("customer_id")
res.show()

+-----------+---------------+-------+-----------+
|customer_id|           name|   city|signup_date|
+-----------+---------------+-------+-----------+
|          1|   Aarav Nguyen| Denver| 2023-01-26|
|          2|Priya Silva-Rao|Seattle| 2023-09-08|
|         41|     Nina Patel| Boston| 2024-12-05|
|         42|    Omar Haddad| Austin| 2024-12-06|
+-----------+---------------+-------+-----------+

**Why / traps:** MERGE fails (or does random things) if the **source has 2 rows for one key**. Dedupe the source first. MERGE is the main tool for upserts in data warehouses.

With Delta Lake or Iceberg tables, Spark SQL has the same `MERGE INTO t USING s ON ...`.

## Q5. Idempotent reload of one slice  ·  🟡 medium

`prac_orders` is a raw copy of `orders` (with duplicates). Reload **October 2025** from a clean source (`SELECT DISTINCT * FROM orders`): first DELETE the slice, then INSERT it again. Run both cells twice: the result must not change. Show `oct_rows` and `oct_orders` for October 2025.

*Expected result: 1 row · columns: `oct_rows`, `oct_orders`*

**How it works:** Delete the whole October slice, then insert the clean October rows. Running both again gives the same rows (idempotent).

In [ ]:
in_oct = (F.col("order_date") >= "2025-10-01") & (F.col("order_date") < "2025-11-01")
new_slice = orders.filter(in_oct).distinct()
prac_orders = orders.filter(~in_oct).unionByName(new_slice)      # replace the slice
res = prac_orders.filter(in_oct).agg(F.count("*").alias("oct_rows"),
                                     F.countDistinct("order_id").alias("oct_orders"))
res.show()

+--------+----------+
|oct_rows|oct_orders|
+--------+----------+
|       4|         4|
+--------+----------+

**Why / traps:** **Delete-then-insert by slice** (date, load id, ...) makes a job **idempotent**: a rerun replaces the slice instead of adding it twice. A plain INSERT on rerun = duplicates.

With files, the same idea is `partitionBy("order_date")` + `spark.sql.sources.partitionOverwriteMode=dynamic` (topic 14).

## Q6. Incremental load with a watermark  ·  🟡 medium

`prac_orders_target` holds clean orders up to 2025-06-30. Load only orders **newer than the latest date already in the target** (the watermark). Then show `num_rows` and `max_date` of the target.

*Expected result: 1 row · columns: `num_rows`, `max_date`*

**How it works:** The watermark = `MAX(order_date)` already in the target. Insert only source rows newer than it.

In [ ]:
target = orders.filter(F.col("order_date") <= "2025-06-30").distinct()
watermark = target.agg(F.max("order_date")).first()[0]
target = target.unionByName(orders.filter(F.col("order_date") > F.lit(watermark)).distinct())
res = target.agg(F.count("*").alias("num_rows"), F.max("order_date").alias("max_date"))
res.show()

+--------+----------+
|num_rows|  max_date|
+--------+----------+
|     104|2025-12-28|
+--------+----------+

**Why / traps:** Weak spot: a row that arrives **late** with an old date is never loaded. Real pipelines use a load timestamp / load id as the watermark, or reload the last N days (idempotent).

## Q7. SCD Type 2: keep the history  ·  🔴 hard

`prac_dim_customer` has one current row per customer (`valid_to = 9999-12-31`, `is_current = 'Y'`). `prac_city_changes` says: customer 1 → Denver, customer 3 → Phoenix (no change!), customer 13 → Austin (old city is NULL). Changes start on **2025-01-15**.

1. **Close** the current row of customers whose city really changed: `valid_to = 2025-01-14`, `is_current = 'N'`.
2. **Insert** a new current row for them from 2025-01-15.
3. Show the rows of customers 1, 3, 13.

*Expected result: 5 rows · columns: `customer_id`, `city`, `valid_from`, `valid_to`, `is_current`*

**How it works:** Step 1 closes the current row of customers whose city really changed (NULL-safe compare). Step 2 inserts a new current row for every change that now has no current row.

In [ ]:
dim = customers.select("customer_id", "city", F.col("signup_date").alias("valid_from"),
                       F.lit("9999-12-31").cast("date").alias("valid_to"), F.lit("Y").alias("is_current"))
changes = spark.createDataFrame([(1, "Denver"), (3, "Phoenix"), (13, "Austin")],
                                "customer_id int, city string")

cur = dim.alias("d").join(changes.alias("s"), "customer_id", "left")
changed = ~F.col("d.city").eqNullSafe(F.col("s.city")) & F.col("s.city").isNotNull()
closed = cur.select(
    "customer_id", "d.city", "d.valid_from",
    F.when(changed, F.lit("2025-01-14").cast("date")).otherwise(F.col("d.valid_to")).alias("valid_to"),
    F.when(changed, "N").otherwise(F.col("d.is_current")).alias("is_current"))
new_rows = (cur.filter(changed)
            .select("customer_id", F.col("s.city").alias("city"),
                    F.lit("2025-01-15").cast("date").alias("valid_from"),
                    F.lit("9999-12-31").cast("date").alias("valid_to"), F.lit("Y").alias("is_current")))
dim = closed.unionByName(new_rows)
res = dim.filter(F.col("customer_id").isin(1, 3, 13)).orderBy("customer_id", "valid_from")
res.show()

+-----------+-------+----------+----------+----------+
|customer_id|   city|valid_from|  valid_to|is_current|
+-----------+-------+----------+----------+----------+
|          1|Chicago|2023-01-26|2025-01-14|         N|
|          1| Denver|2025-01-15|9999-12-31|         Y|
|          3|Phoenix|2023-05-23|9999-12-31|         Y|
|         13|   NULL|2023-10-12|2025-01-14|         N|
|         13| Austin|2025-01-15|9999-12-31|         Y|
+-----------+-------+----------+----------+----------+

**Why / traps:** NULL trap: `s.city <> d.city` is **unknown** when the old city is NULL, so customer 13 would be missed. COALESCE both sides (or a null-safe compare). Customer 3 must **not** get a new row. Order matters: close first, then insert the rows that have no current version.

`eqNullSafe` (SQL `<=>`) is a NULL-safe equals: NULL <=> NULL is true.

## Q8. ALTER TABLE: add a flag column  ·  🟡 medium

Add a column `is_active` (1 letter) to `prac_customers`. Set it to `Y` for customers with at least one order, `N` for the rest. Show `is_active` and `num_customers`.

*Expected result: 2 rows · columns: `is_active`, `num_customers`*

**How it works:** `ALTER TABLE ... ADD` creates the empty column; one UPDATE fills it with CASE + EXISTS; GROUP BY checks the result.

In [ ]:
buyers = orders.select("customer_id").distinct().withColumn("is_active", F.lit("Y"))
prac_customers = (customers.join(buyers, "customer_id", "left")
                  .withColumn("is_active", F.coalesce("is_active", F.lit("N"))))
res = prac_customers.groupBy("is_active").agg(F.count("*").alias("num_customers")).orderBy("is_active")
res.show()

+---------+-------------+
|is_active|num_customers|
+---------+-------------+
|        N|            8|
|        Y|           32|
+---------+-------------+

**Why / traps:** This is a **soft delete** style flag: keep the row, mark it inactive.